In [2]:
%pip install ipywidgets

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\MATIUS AHJ\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


# IndoLens — Pipeline Training & Analisis Dataset Skripsi
Notebook ini dirancang untuk memproses dataset wajah aktor Indonesia secara offline menggunakan akselerasi CPU/GPU dan memuat reference embedding ke runtime IndoLens.

**Urutan Eksekusi:**
1. **Cell 0**: Setup & Pre-Flight Check (verifikasi package, MySQL read-only, dan path file)
2. **Stage 01**: Ekstraksi Wajah (YOLOv8-Face)
3. **Stage 02**: Analisis Matematis Dataset & Perceptual Hashing (dHash)
4. **Stage 03**: Preprocessing Wajah (160x160, Normalisasi VGGFace2)
5. **Stage 04**: Pembuatan Embedding 128-D (FaceNet Inception-ResNet-v1)
6. **Stage 05**: Validasi Embedding & Audit Jarak Euclidean (Intra-class vs Inter-class)
7. **Stage 06**: Integrasi Reference Data ke Runtime Web (`python/embeddings/`)
8. **Stage 07**: Evaluasi Akhir Dataset & Rekapitulasi Metrik Skripsi

In [5]:
# CELL 0: SETUP & PRE-FLIGHT CHECK
import os, sys
from pathlib import Path

_ROOT = Path(os.environ.get('INDOLENS_ROOT', r'C:\laragon\www\IndoLens'))
for _p in [_ROOT, _ROOT / 'training_python', _ROOT / 'python']:
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import torch
import training_python as tp
from training_python import tp_config as cfg
from training_python import tp_database as db

print('[OK] Project Root:', cfg.PROJECT_ROOT)
print('[OK] PyTorch     :', torch.__version__)
print('[OK] GPU / CUDA  :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

records = db.fetch_actor_image_records()
print(f'[OK] MySQL Connected (Read-Only): Ditemukan {len(records)} record gambar aktor terdaftar.')

[OK] Project Root: C:\laragon\www\IndoLens
[OK] PyTorch     : 2.14.1+cu126
[OK] GPU / CUDA  : NVIDIA GeForce RTX 4060 Laptop GPU
[OK] MySQL Connected (Read-Only): Ditemukan 10 record gambar aktor terdaftar.


## Stage 01: Ekstraksi Wajah (YOLOv8-Face)
- Membaca daftar gambar resmi dari MySQL (Read-Only).
- Mendeteksi bounding box wajah menggunakan YOLOv8-Face.
- Menyimpan hasil crop wajah asli ke `artifacts/01_ekstraksi_wajah/crops/`.
- Menangani multi-face (disimpan ke `rejected/` untuk audit) dan zero-face.

In [8]:
# Tips: Anda bisa menguji 1 gambar pertama dengan: tp.run_stage_01(limit=1)
# Jalankan untuk memproses seluruh gambar (aman diinterupsi / resume otomatis):
tp.run_stage_01()

[*] Perangkat Akselerasi YOLO : CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU - 8.0 GB VRAM)
[*] Memuat model YOLO Face dari: C:\laragon\www\IndoLens\storage\app\ai\weights\yolov8n-face.pt

[Stage 01] Mengambil daftar gambar aktor resmi (MySQL Read-Only)...
[*] Berhasil memuat 10 record gambar aktor resmi dari MySQL.

  MEMULAI DETEKSI & CROP WAJAH SATU PER SATU (YOLOv8-FACE)

[1/10] Gambar ID #1 | Aktor: Bayu Eko Moektito (#1)
    Path: actor-images/1-bayu-skak/bayu-skak-01.jpg
    [OK] Sukses Crop: 86x116 px (conf: 0.9656) -> 1.png

[2/10] Gambar ID #5 | Aktor: Bayu Eko Moektito (#1)
    Path: actor-images/1-bayu-skak/bayu-skak-05.jpg
    [!] Multi-face terdeteksi: ditemukan 2 wajah. Wajah ambigu ditolak agar tidak salah identitas.

[3/10] Gambar ID #6 | Aktor: Bayu Eko Moektito (#1)
    Path: actor-images/1-bayu-skak/bayu-skak-06.jpg
    [OK] Sukses Crop: 80x106 px (conf: 0.9490) -> 6.png

[4/10] Gambar ID #7 | Aktor: Bayu Eko Moektito (#1)
    Path: actor-images/1-bayu-skak/bayu-skak-

{'total_queued': 10,
 'skipped_existing': 0,
 'success': 8,
 'zero_face': 0,
 'multi_face': 2,
 'unreadable': 0,
 'error': 0}

## Stage 02: Analisis Matematis Dataset
- Menghitung statistik dimensi gambar asli & crop wajah (min, max, mean, std).
- Menghitung distribusi skor confidence deteksi YOLO.
- Mendeteksi duplikasi visual dengan perceptual difference hashing (dHash 64-bit).

In [11]:
tp.run_stage_02()


  STAGE 02: ANALISIS MATEMATIS DATASET WAJAH INDOLENS
  LAPORAN ANALISIS MATEMATIS DATASET INDOLENS (SKRIPSI)
Total Data Terdaftar di MySQL     : 10
  - Berhasil Diekstrak (1 Wajah)  : 8 (80.0%)
  - Nol Wajah Terdeteksi          : 0
  - Multi-Face (Wajah Ambigu)     : 2
  - Gambar Rusak / Tidak Terbaca  : 0
  - Error Teknis                  : 0

---------------------------------------------------------------------------
1. STATISTIK RESOLUSI FACE-CROP WAJAH (Piksel):
  - Lebar  : Min=73.0 | Max=151.0 | Mean=95.625 | Std=23.3235
  - Tinggi : Min=101.0 | Max=189.0 | Mean=126.125 | Std=27.0945
  - Rasio W/H : Mean=0.7548 (Standar wajah ideal mendekati 1.0)

---------------------------------------------------------------------------
2. STATISTIK CONFIDENCE YOLOv8-FACE:
  - Mean Confidence : 0.9411
  - Min Confidence  : 0.8893
  - Max Confidence  : 0.9686
  - Standar Deviasi : 0.0271

---------------------------------------------------------------------------
3. EVALUASI DUPLIKASI GAMBAR (

{'generated_at': 1792388262.8137527,
 'total_source_records': 10,
 'status_distribution': {'success': 8, 'multi_face': 2},
 'per_actor_breakdown': [{'actor_id': 1,
   'full_name': 'Bayu Eko Moektito',
   'total': 10,
   'success': 8,
   'zero_face': 0,
   'multi_face': 2,
   'unreadable': 0,
   'error': 0}],
 'original_image_resolution': {'width': {'count': 10,
   'min': 300.0,
   'max': 1200.0,
   'mean': 605.7,
   'median': 573.0,
   'std': 257.1673},
  'height': {'count': 10,
   'min': 208.0,
   'max': 998.0,
   'mean': 450.9,
   'median': 390.0,
   'std': 221.5344}},
 'face_crop_resolution': {'width': {'count': 8,
   'min': 73.0,
   'max': 151.0,
   'mean': 95.625,
   'median': 89.5,
   'std': 23.3235},
  'height': {'count': 8,
   'min': 101.0,
   'max': 189.0,
   'mean': 126.125,
   'median': 119.5,
   'std': 27.0945},
  'aspect_ratio': {'count': 8,
   'min': 0.7154,
   'max': 0.7989,
   'mean': 0.7548,
   'median': 0.7604,
   'std': 0.0257}},
 'yolo_detection_confidence': {'count

## Stage 03: Preprocessing Wajah
- Konversi BGR -> RGB.
- Resize presisi ke 160x160 piksel menggunakan bilinear interpolation.
- Normalisasi float32 VGGFace2 standardization: `(x - mean) / std`.
- Menyimpan tensor ke `artifacts/03_preprocessing_wajah/tensors/`.

In [14]:
tp.run_stage_03()


  STAGE 03: PREPROCESSING WAJAH UNTUK FACENET (160x160 RGB NORMALIZED)
[*] Ditemukan 8 crop wajah valid dari Stage 01.
[1/8] Preprocessing Image #1 | Bayu Eko Moektito
    [OK] Sukses: Tensor (160,160,3) -> 1.npy
[2/8] Preprocessing Image #6 | Bayu Eko Moektito
    [OK] Sukses: Tensor (160,160,3) -> 6.npy
[3/8] Preprocessing Image #7 | Bayu Eko Moektito
    [OK] Sukses: Tensor (160,160,3) -> 7.npy
[4/8] Preprocessing Image #8 | Bayu Eko Moektito
    [OK] Sukses: Tensor (160,160,3) -> 8.npy
[5/8] Preprocessing Image #10 | Bayu Eko Moektito
    [OK] Sukses: Tensor (160,160,3) -> 10.npy
[6/8] Preprocessing Image #11 | Bayu Eko Moektito
    [OK] Sukses: Tensor (160,160,3) -> 11.npy
[7/8] Preprocessing Image #12 | Bayu Eko Moektito
    [OK] Sukses: Tensor (160,160,3) -> 12.npy
[8/8] Preprocessing Image #13 | Bayu Eko Moektito
    [OK] Sukses: Tensor (160,160,3) -> 13.npy

  STAGE 03 SELESAI
  Durasi Total        : 0.04 detik
  Berhasil Diproses   : 8
  Dilewati (Existing) : 0
  Error      

{'success': 8, 'skipped': 0, 'error': 0}

## Stage 04: Pembuatan Embedding 128-D
- Inferensi model neural network FaceNet Inception-ResNet-v1 (pretrained VGGFace2).
- Truncate ke 128 dimensi & normalisasi Euclidean L2-Norm (unit length = 1.0).
- Menyimpan vektor embedding `.npy` ke `artifacts/04_pembuatan_embedding/vectors/`.

In [17]:
tp.run_stage_04()


  STAGE 04: PEMBUATAN EMBEDDING 128-D (FACENET INCEPTION-RESNET-V1)
[*] Perangkat Inferensi: CUDA GPU: NVIDIA GeForce RTX 4060 Laptop GPU (VRAM: 8.0 GB)
[*] Menginisialisasi FaceNet InceptionResnetV1 (pretrained='vggface2')...
[OK] FaceNet siap di: cuda
[*] Ditemukan 8 tensor wajah valid dari Stage 03.
[1/8] Extracting Embedding Image #1 | Bayu Eko Moektito
    [OK] Sukses Ekstraksi (512-D & 128-D, 83.49 ms) -> 1.npy
[2/8] Extracting Embedding Image #6 | Bayu Eko Moektito
    [OK] Sukses Ekstraksi (512-D & 128-D, 32.79 ms) -> 6.npy
[3/8] Extracting Embedding Image #7 | Bayu Eko Moektito
    [OK] Sukses Ekstraksi (512-D & 128-D, 32.72 ms) -> 7.npy
[4/8] Extracting Embedding Image #8 | Bayu Eko Moektito
    [OK] Sukses Ekstraksi (512-D & 128-D, 32.84 ms) -> 8.npy
[5/8] Extracting Embedding Image #10 | Bayu Eko Moektito
    [OK] Sukses Ekstraksi (512-D & 128-D, 33.67 ms) -> 10.npy
[6/8] Extracting Embedding Image #11 | Bayu Eko Moektito
    [OK] Sukses Ekstraksi (512-D & 128-D, 32.71 ms)

{'success': 8,
 'skipped': 0,
 'error': 0,
 'device': 'CUDA GPU: NVIDIA GeForce RTX 4060 Laptop GPU (VRAM: 8.0 GB)'}

## Stage 05: Validasi Kualitas Embedding & Audit Jarak Euclidean
- Validasi bebas NaN, Inf, dimensi tepat 128, dan L2-norm == 1.0.
- Menghitung matriks jarak Euclidean intra-class (foto aktor yang sama).
- Menandai pasangan foto dengan variasi sudut/pencahayaan ekstrem (anomali).

In [19]:
tp.run_stage_05()


  STAGE 05: VALIDASI EMBEDDING 128-D & AUDIT JARAK EUCLIDEAN
[*] Total vektor diperiksa : 8
[*] Lolos validasi         : 8
[*] Gagal validasi         : 0

HASIL KALIBRASI THRESHOLD & AUDIT MATEMATIS SKRIPSI:
1. Aktor Terdaftar               : 1 aktor
2. Metode Kalibrasi Threshold    : INTRA_CLASS_STATISTICAL_CI95 (mu + 2*sigma)
3. THRESHOLD OPTIMAL TERKALIBRASI: 0.85 (Bukan Hardcoded)
4. Estimasi Akurasi Kalibrasi    : 92.86%
5. Intra-Class Distance (Kemiripan Foto Aktor yang Sama):
   - Rata-rata Jarak (Mean)      : 0.628
   - Standar Deviasi (Std)       : 0.1144
   - Jarak Terkecil / Terbesar   : 0.466 / 0.9114

---------------------------------------------------------------------------
TABEL KOMPARASI ILMIAH DIMENSI FITUR (BAHAN SIDANG SKRIPSI):
Dimensi    | Intra Mean   | Inter Mean   | Margin     | Optimal Th  
-----------------------------------------------------------------
512-D      | 0.628        | 0.0          | 0.372      | 0.8568      
128-D      | 0.612        | 0.0     

{'generated_at': 1792388267.0488229,
 'primary_dimension': 512,
 'total_embeddings_inspected': 8,
 'valid_embeddings_count': 8,
 'validation_failures': [],
 'registered_actors_count': 1,
 'calibrated_threshold': 0.85,
 'calibration_method': 'INTRA_CLASS_STATISTICAL_CI95 (mu + 2*sigma)',
 'estimated_accuracy_pct': 92.86,
 'intra_class_distance': {'count': 28,
  'mean': 0.628,
  'min': 0.466,
  'max': 0.9114,
  'std': 0.1144,
  'pass_threshold_ratio': 0.6786},
 'inter_class_distance': {'count': 0,
  'mean': None,
  'min': None,
  'max': None,
  'std': None,
  'safe_threshold_ratio': None},
 'cross_actor_available': False,
 'flagged_anomalies_count': 9}

## Stage 06: Integrasi Reference Data ke Runtime Web
- Membuat staging data terisolasi & backup otomatis direktori `python/embeddings/`.
- Mempromosikan vektor reference data yang lolos validasi ke folder runtime `python/embeddings/<actor_slug>/`.
- Mengirim sinyal reload memori ke service IndoLens (jika service sedang aktif).

In [23]:
tp.run_stage_06(promote_to_production=True)


  STAGE 06: INTEGRASI REFERENCE DATA KE SISTEM WEB INDOLENS
[*] Memuat 8 vektor embedding terverifikasi dari Stage 05.
[OK] Staging selesai: 8 vektor untuk 1 aktor di:
    C:\laragon\www\IndoLens\training_python\artifacts\06_integrasi_reference_data\staging
[*] Membuat backup folder runtime ke: C:\laragon\www\IndoLens\training_python\artifacts\06_integrasi_reference_data\backups\backup_1792388267
[*] Bayu Eko Moektito: 2 embedding stale dibuang dari runtime -> ['5.npy', '9.npy']
[OK] Berhasil dipromosikan ke runtime: C:\laragon\www\IndoLens\python\embeddings
[OK] Threshold runtime diselaraskan : 0.85 (INTRA_CLASS_STATISTICAL_CI95 (mu + 2*sigma))
[OK] Dimensi embedding diselaraskan : 512-D
[OK] python/config/dataset.py diselaraskan (DIM=512, TH=0.85)
[-] Catatan reload runtime: Microservice tidak aktif atau tidak dapat dijangkau: <urlopen error [WinError 10061] No connection could be made because the target machine actively refused it>
    (Normal jika yolo_service sedang tidak berjala

{'integrated_at': 1792388269.510405,
 'staged_count': 8,
 'actors_integrated': ['bayu_eko_moektito'],
 'pruned_stale_embeddings': {'bayu_eko_moektito': ['5.npy', '9.npy']},
 'backup_location': 'C:\\laragon\\www\\IndoLens\\training_python\\artifacts\\06_integrasi_reference_data\\backups\\backup_1792388267',
 'production_target': 'C:\\laragon\\www\\IndoLens\\python\\embeddings',
 'service_reload_success': False,
 'service_reload_detail': 'Microservice tidak aktif atau tidak dapat dijangkau: <urlopen error [WinError 10061] No connection could be made because the target machine actively refused it>'}

## Stage 07: Evaluasi Akhir Dataset & Rekapitulasi Metrik Skripsi
- Rekapitulasi seluruh metrik penelitian (deteksi YOLO, distribusi crop, audit jarak).
- Evaluasi opsional terhadap folder `probe_set/` jika disediakan foto uji independen.
- Menghasilkan laporan resmi `evaluation_report.json` dan `evaluation_report.txt`.

In [25]:
tp.run_stage_07()


  STAGE 07: EVALUASI AKHIR PIPELINE DATASET & METRIK SKRIPSI
  LAPORAN EVALUASI AKHIR PIPELINE TRAINING INDOLENS (SKRIPSI)
Metode Deteksi Wajah   : YOLOv8-Face (conf=0.45)
Model Ekstraksi Fitur  : FaceNet InceptionResnetV1 (vggface2)
Dimensi Vektor Acuan   : 512-D (L2-Normalized)
Metrik Jarak & Nilai   : Euclidean Distance (L2 Norm)
Threshold Terkalibrasi : <= 0.85 (Metode: INTRA_CLASS_STATISTICAL_CI95 (mu + 2*sigma))
Perangkat Komputasi    : CUDA GPU: NVIDIA GeForce RTX 4060 Laptop GPU (VRAM: 8.0 GB)

---------------------------------------------------------------------------
1. METRIK DATA PIPELINE (REAL COMPUTATION):
  - Total Record MySQL          : 10
  - Gambar Terbaca              : 10
  - Gambar Rusak / Hilang       : 0
  - Zero Face (Wajah Luput)     : 0
  - Multi Face (Wajah Ambigu)   : 2
  - Wajah Berhasil Di-crop      : 8
  - Vektor Embedding Valid      : 8
  - Aktor Terdaftar             : 1 aktor

--------------------------------------------------------------------------

{'evaluation_timestamp': 1792388269.5596163,
 'methodology': {'detector': 'YOLOv8-Face (conf=0.45)',
  'feature_extractor': 'FaceNet InceptionResnetV1 (vggface2)',
  'embedding_dimension': 512,
  'metric': 'Euclidean Distance (L2 Norm)',
  'threshold': 0.65},
 'pipeline_pipeline_summary': {'total_mysql_records': 10,
  'readable_images': 10,
  'unreadable_images': 0,
  'zero_face_images': 0,
  'multi_face_images': 2,
  'valid_crops_generated': 8,
  'valid_tensors_preprocessed': 8,
  'valid_embeddings_generated': 8,
  'verified_active_embeddings': 8,
  'registered_actors_count': 1,
  'execution_device': 'CUDA GPU: NVIDIA GeForce RTX 4060 Laptop GPU (VRAM: 8.0 GB)'},
 'mathematical_audit': {'intra_class_distance': {'count': 28,
   'mean': 0.628,
   'min': 0.466,
   'max': 0.9114,
   'std': 0.1144,
   'pass_threshold_ratio': 0.6786},
  'inter_class_distance': {'count': 0,
   'mean': None,
   'min': None,
   'max': None,
   'std': None,
   'safe_threshold_ratio': None}},
 'independent_probe